# Step 3D.1 - Mid KL Same Subject Stress

This notebook stays on `dev_tune_200` and reuses the same derived stress input created by Step 3D.

Purpose:

```text
Test whether lower/mid-guidance subject-gated methods can avoid same-subject
target over-injection while keeping useful rewrite/paraphrase performance.
```

It does not create a new split and does not touch:

```text
analysis_500
ablation_500
final_test_500
final_test_full
```

In [ ]:
%%bash
set -euo pipefail

pip install -q   "transformers==4.46.3"   "datasets>=4.0.0"   "accelerate>=1.11.0"   "sentencepiece>=0.2.0"   "packaging"   "bitsandbytes>=0.43.0"

In [ ]:
import sys
import subprocess
from pathlib import Path
import torch, transformers, datasets, accelerate

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"
STRESS_INPUT = RUN_ROOT / "same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl"
STRESS_SUMMARY = RUN_ROOT / "same_subject_stress_inputs/summary.json"
STEP3D_REPORT = RUN_ROOT / "dev_tune_200_same_subject_stress_report_v1"
DENSE_REPORT = RUN_ROOT / "dev_tune_200_dense_pareto_subject_v1"
OUT_REPORT = RUN_ROOT / "dev_tune_200_same_subject_stress_midkl_v1"

print("Python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
except Exception as exc:
    commit = f"unavailable: {exc!r}"
print("git commit:", commit)
print("project dir:", PROJECT_DIR)
print("protocol_version:", "counterfact_direction1_v1")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Save Runtime Environment

In [ ]:
import json
from pathlib import Path
import subprocess
import torch, transformers, datasets, accelerate

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"
assert PROJECT_DIR.exists(), f"Missing project dir after Drive mount: {PROJECT_DIR}"
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
except Exception as exc:
    commit = f"unavailable: {exc!r}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "git_commit": commit,
}
out = RUN_ROOT / "runtime_environment_step3d1.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)
print("Wrote:", out)
print(json.dumps(env, indent=2))

## Preflight Validation

In [ ]:
import csv
import json
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"
STRESS_INPUT = RUN_ROOT / "same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl"
STRESS_SUMMARY = RUN_ROOT / "same_subject_stress_inputs/summary.json"
STEP3D_REPORT = RUN_ROOT / "dev_tune_200_same_subject_stress_report_v1"
DENSE_REPORT = RUN_ROOT / "dev_tune_200_dense_pareto_subject_v1"
OUT_REPORT = RUN_ROOT / "dev_tune_200_same_subject_stress_midkl_v1"

required_paths = [
    STRESS_INPUT,
    STRESS_SUMMARY,
    STEP3D_REPORT / "report_summary.json",
    STEP3D_REPORT / "same_subject_stress_summary.csv",
    DENSE_REPORT / "report_summary.json",
    DENSE_REPORT / "dense_pareto_summary.csv",
    DENSE_REPORT / "feasible_ranking.csv",
    DENSE_REPORT / "best_by_kl_budget.csv",
    DENSE_REPORT / "best_guided_by_kl_budget.csv",
]
for path in required_paths:
    assert path.exists(), f"Missing required artifact: {path}"

stress_summary = json.load(STRESS_SUMMARY.open())
assert stress_summary["num_edits"] == 200
assert stress_summary["analysis_500_used"] is False
assert stress_summary["final_test_used"] is False
counts = stress_summary["stress_counts"]
assert int(counts.get("same_subject_template", 0)) == 600, counts
assert int(counts.get("generation", 0)) == 400, counts

step3d = json.load((STEP3D_REPORT / "report_summary.json").open())
assert "target_new" in step3d["stress_definition"]
assert "over-injection" in step3d["stress_definition"]
assert step3d["analysis_500_used"] is False
assert step3d["final_test_used"] is False
assert "stress_tfpr_budget" in step3d["stress_budget_rule"]

dense = json.load((DENSE_REPORT / "report_summary.json").open())
assert dense["split_role"] == "dev_tune_200"
assert dense["gate_mode"] == "subject"
assert dense.get("kl_budgets") == [1, 2, 4, 6, 8, 10]
print("Preflight validation OK.")

## Target Validation

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"
STRESS_INPUT = RUN_ROOT / "same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from llada_counterfact_protocol import normalize_counterfact_text

rows = [json.loads(line) for line in STRESS_INPUT.open("r", encoding="utf-8") if line.strip()]
assert len(rows) == 200
for row in rows:
    requested = row.get("requested_rewrite") or {}
    expected = None
    if isinstance(requested, dict):
        expected = ((requested.get("target_new") or {}).get("str"))
    if expected is None and isinstance(row.get("target_new"), dict):
        expected = row["target_new"].get("text") or row["target_new"].get("str")
    assert expected is not None, f"Missing target_new for {row['id']}"
    assert normalize_counterfact_text(row["target"]) == normalize_counterfact_text(expected), (
        f"Stress target is not target_new for {row['id']}: {row['target']!r} vs {expected!r}"
    )
print("Target-new over-injection semantics validated for", len(rows), "stress rows.")

## Candidate Registry

In [ ]:
RUN_ROOT = Path("/content/drive/MyDrive/Masters/SB V2/SB/runs/counterfact_direction1_v1")
ALLOW_OVERWRITE = False

CANDIDATES_NEW = [
    {
        "label": "myopic_score_gated_subject_gs1.5",
        "method": "myopic_score_gated_subject",
        "family": "myopic_score",
        "guidance_scale": 1.5,
        "mc_rollouts": 0,
        "output_dir": "dev_tune_200_same_subject_stress_myopic_gs150",
    },
    {
        "label": "myopic_score_gated_subject_gs1.75",
        "method": "myopic_score_gated_subject",
        "family": "myopic_score",
        "guidance_scale": 1.75,
        "mc_rollouts": 0,
        "output_dir": "dev_tune_200_same_subject_stress_myopic_gs175",
    },
    {
        "label": "mc_bridge_gated_subject_gs1.5",
        "method": "mc_bridge_gated_subject",
        "family": "mc_bridge",
        "guidance_scale": 1.5,
        "mc_rollouts": 2,
        "output_dir": "dev_tune_200_same_subject_stress_mc_bridge_gs150",
    },
    {
        "label": "mc_bridge_gated_subject_gs1.75",
        "method": "mc_bridge_gated_subject",
        "family": "mc_bridge",
        "guidance_scale": 1.75,
        "mc_rollouts": 2,
        "output_dir": "dev_tune_200_same_subject_stress_mc_bridge_gs175",
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs1.5",
        "method": "no_rollout_bridge_gated_subject",
        "family": "no_rollout_bridge",
        "guidance_scale": 1.5,
        "mc_rollouts": 0,
        "output_dir": "dev_tune_200_same_subject_stress_no_rollout_gs150",
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs1.75",
        "method": "no_rollout_bridge_gated_subject",
        "family": "no_rollout_bridge",
        "guidance_scale": 1.75,
        "mc_rollouts": 0,
        "output_dir": "dev_tune_200_same_subject_stress_no_rollout_gs175",
    },
]
print(json.dumps(CANDIDATES_NEW, indent=2))

## Overwrite Guard

In [ ]:
for candidate in CANDIDATES_NEW:
    out = RUN_ROOT / candidate["output_dir"]
    if out.exists() and not ALLOW_OVERWRITE:
        raise RuntimeError(f"Output already exists: {out}")
if OUT_REPORT.exists() and not ALLOW_OVERWRITE:
    raise RuntimeError(f"Report directory already exists: {OUT_REPORT}")
print("Overwrite guard passed.")

## Run myopic_score_gated_subject_gs1.5

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_myopic_gs150

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_myopic_gs150 \
  --methods myopic_score_gated_subject \
  --split_role dev_tune_200 \
  --stress_eval 1 \
  --stress_name same_subject_locality_midkl \
  --target_semantics target_new_over_injection \
  --analysis_500_used 0 \
  --final_test_used 0 \
  --protocol_version counterfact_direction1_v1 \
  --edit_access given_at_edit_time \
  --training_access none \
  --hyperparameter_access dev_tune_only \
  --gate_mode subject \
  --guidance_scale 1.5 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --steps 4 \
  --eval_samples 4 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_myopic_gs150/stdout.log

## Run myopic_score_gated_subject_gs1.75

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_myopic_gs175

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_myopic_gs175 \
  --methods myopic_score_gated_subject \
  --split_role dev_tune_200 \
  --stress_eval 1 \
  --stress_name same_subject_locality_midkl \
  --target_semantics target_new_over_injection \
  --analysis_500_used 0 \
  --final_test_used 0 \
  --protocol_version counterfact_direction1_v1 \
  --edit_access given_at_edit_time \
  --training_access none \
  --hyperparameter_access dev_tune_only \
  --gate_mode subject \
  --guidance_scale 1.75 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --steps 4 \
  --eval_samples 4 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_myopic_gs175/stdout.log

## Run mc_bridge_gated_subject_gs1.5

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_gs150

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_gs150 \
  --methods mc_bridge_gated_subject \
  --split_role dev_tune_200 \
  --stress_eval 1 \
  --stress_name same_subject_locality_midkl \
  --target_semantics target_new_over_injection \
  --analysis_500_used 0 \
  --final_test_used 0 \
  --protocol_version counterfact_direction1_v1 \
  --edit_access given_at_edit_time \
  --training_access none \
  --hyperparameter_access dev_tune_only \
  --gate_mode subject \
  --guidance_scale 1.5 \
  --bridge_topk 4 \
  --mc_rollouts 2 \
  --steps 4 \
  --eval_samples 4 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_gs150/stdout.log

## Run mc_bridge_gated_subject_gs1.75

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_gs175

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_gs175 \
  --methods mc_bridge_gated_subject \
  --split_role dev_tune_200 \
  --stress_eval 1 \
  --stress_name same_subject_locality_midkl \
  --target_semantics target_new_over_injection \
  --analysis_500_used 0 \
  --final_test_used 0 \
  --protocol_version counterfact_direction1_v1 \
  --edit_access given_at_edit_time \
  --training_access none \
  --hyperparameter_access dev_tune_only \
  --gate_mode subject \
  --guidance_scale 1.75 \
  --bridge_topk 4 \
  --mc_rollouts 2 \
  --steps 4 \
  --eval_samples 4 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_mc_bridge_gs175/stdout.log

## Run no_rollout_bridge_gated_subject_gs1.5

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_no_rollout_gs150

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_no_rollout_gs150 \
  --methods no_rollout_bridge_gated_subject \
  --split_role dev_tune_200 \
  --stress_eval 1 \
  --stress_name same_subject_locality_midkl \
  --target_semantics target_new_over_injection \
  --analysis_500_used 0 \
  --final_test_used 0 \
  --protocol_version counterfact_direction1_v1 \
  --edit_access given_at_edit_time \
  --training_access none \
  --hyperparameter_access dev_tune_only \
  --gate_mode subject \
  --guidance_scale 1.5 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --steps 4 \
  --eval_samples 4 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_no_rollout_gs150/stdout.log

## Run no_rollout_bridge_gated_subject_gs1.75

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_no_rollout_gs175

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_no_rollout_gs175 \
  --methods no_rollout_bridge_gated_subject \
  --split_role dev_tune_200 \
  --stress_eval 1 \
  --stress_name same_subject_locality_midkl \
  --target_semantics target_new_over_injection \
  --analysis_500_used 0 \
  --final_test_used 0 \
  --protocol_version counterfact_direction1_v1 \
  --edit_access given_at_edit_time \
  --training_access none \
  --hyperparameter_access dev_tune_only \
  --gate_mode subject \
  --guidance_scale 1.75 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --steps 4 \
  --eval_samples 4 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --target_logit_bias 0.0 \
  --skip_candidate_coverage 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_same_subject_stress_no_rollout_gs175/stdout.log

## Config And Completeness Validation

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"

CARRY_FORWARD_EXPECTED = [
    {
        "label": "base",
        "method": "base",
        "run_dir": "dev_tune_200_same_subject_stress_base",
        "gate_required": False,
    },
    {
        "label": "prompt_memory_gated_subject_gs1.0",
        "method": "prompt_memory_gated_subject",
        "run_dir": "dev_tune_200_same_subject_stress_prompt_memory_subject",
        "gate_required": True,
    },
    {
        "label": "myopic_score_gated_subject_gs2.0",
        "method": "myopic_score_gated_subject",
        "run_dir": "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200",
        "gate_required": True,
    },
    {
        "label": "mc_bridge_gated_subject_gs2.0",
        "method": "mc_bridge_gated_subject",
        "run_dir": "dev_tune_200_same_subject_stress_mc_bridge_subject_gs200",
        "gate_required": True,
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs2.0",
        "method": "no_rollout_bridge_gated_subject",
        "run_dir": "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200",
        "gate_required": True,
    },
]

NEW_EXPECTED = [
    {
        "label": candidate["label"],
        "method": candidate["method"],
        "run_dir": candidate["output_dir"],
        "gate_required": True,
        "stress_eval_required": True,
        "expected_guidance_scale": candidate["guidance_scale"],
        "expected_mc_rollouts": candidate["mc_rollouts"],
    }
    for candidate in CANDIDATES_NEW
]

EXPECTED_ALL = CARRY_FORWARD_EXPECTED + NEW_EXPECTED


def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def stress_bucket(prompt_id):
    prompt_id = str(prompt_id)
    if "_same_subject_template_" in prompt_id:
        return "same_subject_template"
    if "_generation_" in prompt_id:
        return "generation"
    if "_attribute_" in prompt_id:
        return "attribute"
    return "unknown"


def mean(values):
    values = [float(v) for v in values if v is not None]
    return sum(values) / len(values) if values else None


def validate_stress_run(spec):
    run_dir = RUN_ROOT / spec["run_dir"]
    for artifact in ["run_config.json", "summary.json", "per_case_results.jsonl"]:
        assert (run_dir / artifact).exists(), f"Missing {artifact} for {spec['label']}: {run_dir}"

    cfg = json.load((run_dir / "run_config.json").open())
    assert cfg["protocol_version"] == "counterfact_direction1_v1"
    assert cfg["split_role"] == "dev_tune_200"
    assert cfg["methods"] == [spec["method"]] or spec["method"] in cfg["methods"]
    assert cfg.get("analysis_500_used", False) is False
    assert cfg.get("final_test_used", False) is False
    if spec.get("stress_eval_required"):
        assert cfg["stress_eval"] is True
        assert cfg["stress_name"] == "same_subject_locality_midkl"
        assert cfg["target_semantics"] == "target_new_over_injection"
        assert cfg["edit_access"] == "given_at_edit_time"
        assert cfg["training_access"] == "none"
        assert cfg["hyperparameter_access"] == "dev_tune_only"
        assert cfg["rollout_config"]["gate_mode"] == "subject"
        assert abs(float(cfg["rollout_config"]["guidance_scale"]) - spec["expected_guidance_scale"]) < 1e-8
        assert int(cfg["rollout_config"]["mc_rollouts"]) == int(spec["expected_mc_rollouts"])

    rows = list(read_jsonl(run_dir / "per_case_results.jsonl"))
    by_bucket = defaultdict(list)
    for row in rows:
        method = row.get("method_variant") or row.get("method")
        if method != spec["method"] or row.get("bucket") != "near_locality":
            continue
        by_bucket[stress_bucket(row.get("prompt_id"))].append(row)

    expected = {
        "same_subject_template": (200, 600),
        "generation": (200, 400),
    }
    for bucket, (expected_edits, expected_rows) in expected.items():
        bucket_rows = by_bucket[bucket]
        edit_ids = {str(row.get("edit_id") or row.get("case_id")) for row in bucket_rows}
        assert len(edit_ids) == expected_edits, (spec["label"], bucket, len(edit_ids))
        assert len(bucket_rows) == expected_rows, (spec["label"], bucket, len(bucket_rows))
        if spec["gate_required"]:
            activation = mean(row.get("gate_activation_rate") for row in bucket_rows)
            assert activation is not None and activation >= 0.95, (spec["label"], bucket, activation)
    print("Config and completeness OK:", spec["label"])


for spec in EXPECTED_ALL:
    validate_stress_run(spec)
print("All Step 3D.1 new and carry-forward stress configs and row completeness checks OK.")

## Build Merged Mid-KL Stress Report

In [ ]:
import csv
import json
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"
OUT_REPORT = RUN_ROOT / "dev_tune_200_same_subject_stress_midkl_v1"
OUT_REPORT.mkdir(parents=True, exist_ok=True)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from llada_experiment_reports import paired_bootstrap_delta_by_case

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def mean_or_none(values):
    values = [float(v) for v in values if v is not None]
    return sum(values) / len(values) if values else None

def write_csv(path, rows):
    if not rows:
        path.write_text("")
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def stress_bucket(prompt_id):
    prompt_id = str(prompt_id)
    if "_same_subject_template_" in prompt_id:
        return "same_subject_template"
    if "_generation_" in prompt_id:
        return "generation"
    if "_attribute_" in prompt_id:
        return "attribute"
    return "unknown"

CARRY_FORWARD = [
    {
        "label": "base",
        "method": "base",
        "family": "base",
        "guidance_scale": 0.0,
        "run_dir": "dev_tune_200_same_subject_stress_base",
    },
    {
        "label": "prompt_memory_gated_subject_gs1.0",
        "method": "prompt_memory_gated_subject",
        "family": "prompt_memory",
        "guidance_scale": 1.0,
        "run_dir": "dev_tune_200_same_subject_stress_prompt_memory_subject",
    },
    {
        "label": "myopic_score_gated_subject_gs2.0",
        "method": "myopic_score_gated_subject",
        "family": "myopic_score",
        "guidance_scale": 2.0,
        "run_dir": "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200",
    },
    {
        "label": "mc_bridge_gated_subject_gs2.0",
        "method": "mc_bridge_gated_subject",
        "family": "mc_bridge",
        "guidance_scale": 2.0,
        "run_dir": "dev_tune_200_same_subject_stress_mc_bridge_subject_gs200",
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs2.0",
        "method": "no_rollout_bridge_gated_subject",
        "family": "no_rollout_bridge",
        "guidance_scale": 2.0,
        "run_dir": "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200",
    },
]
NEW_SPECS = [
    {**candidate, "run_dir": candidate["output_dir"]}
    for candidate in CANDIDATES_NEW
]
RUN_SPECS = CARRY_FORWARD + NEW_SPECS

dense_path = RUN_ROOT / "dev_tune_200_dense_pareto_subject_v1/dense_pareto_summary.csv"
dense_by_label = {}
with dense_path.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        dense_by_label[row["label"]] = row

for spec in RUN_SPECS:
    label = spec["label"]
    if label == "base":
        continue
    assert label in dense_by_label, f"Missing dense Pareto metrics for {label}"
    for key in [
        "rewrite_exact",
        "declarative_paraphrases_exact",
        "selection_score",
        "primary_sparse_guidance_kl",
    ]:
        assert dense_by_label[label].get(key) not in {None, ""}, f"Missing {key} for {label}"

method_rows = {}
summary_rows = []
per_label_bucket = defaultdict(dict)
efficiency_by_label = {}

for spec in RUN_SPECS:
    run_dir = RUN_ROOT / spec["run_dir"]
    all_rows = [
        {**row, "method_variant": spec["label"]}
        for row in read_jsonl(run_dir / "per_case_results.jsonl")
        if (row.get("method_variant") or row.get("method")) == spec["method"]
    ]
    method_rows[spec["label"]] = all_rows
    run_summary = json.load((run_dir / "summary.json").open())
    efficiency = run_summary.get("efficiency", {})
    method_count = len({row.get("method_variant") or row.get("method") for row in read_jsonl(run_dir / "per_case_results.jsonl")}) or 1
    wall = float(efficiency.get("wall_time_seconds") or 0.0)
    efficiency_by_label[spec["label"]] = {
        "gpu_minutes_per_edit": wall / 60.0 / 200.0 / method_count if wall else None,
        "model_evals_per_edit": (float(efficiency.get("model_evals_per_edit") or 0.0) / method_count) if efficiency.get("model_evals_per_edit") else None,
    }
    grouped = defaultdict(list)
    for row in all_rows:
        if row.get("bucket") != "near_locality":
            continue
        grouped[(stress_bucket(row.get("prompt_id")), str(row.get("edit_id") or row.get("case_id")))].append(row)
    by_stress = defaultdict(list)
    for (bucket, edit_id), rows in grouped.items():
        by_stress[bucket].append({
            "edit_id": edit_id,
            "target_false_positive_rate": mean_or_none(row.get("target_false_positive_rate") for row in rows),
            "target_new_exact_rate": mean_or_none(row.get("exact_rate") for row in rows),
            "target_new_token_f1": mean_or_none(row.get("token_f1") for row in rows),
            "malformed_rate": mean_or_none(row.get("malformed_rate") for row in rows),
            "sparse_guidance_kl": mean_or_none(row.get("sparse_guidance_kl") for row in rows),
            "gate_activation_rate": mean_or_none(row.get("gate_activation_rate") for row in rows),
            "prompt_rows": len(rows),
        })
    for bucket, values in sorted(by_stress.items()):
        row = {
            "label": spec["label"],
            "family": spec["family"],
            "guidance_scale": spec["guidance_scale"],
            "stress_bucket": bucket,
            "num_edits": len(values),
            "num_prompt_rows": sum(item["prompt_rows"] for item in values),
            "gate_activation_rate": mean_or_none(item["gate_activation_rate"] for item in values),
            "target_false_positive_rate": mean_or_none(item["target_false_positive_rate"] for item in values),
            "target_new_exact_rate": mean_or_none(item["target_new_exact_rate"] for item in values),
            "target_new_token_f1": mean_or_none(item["target_new_token_f1"] for item in values),
            "malformed_rate": mean_or_none(item["malformed_rate"] for item in values),
            "mean_sparse_guidance_kl": mean_or_none(item["sparse_guidance_kl"] for item in values),
            **efficiency_by_label[spec["label"]],
        }
        summary_rows.append(row)
        per_label_bucket[spec["label"]][bucket] = row

base_by_bucket = per_label_bucket["base"]
for row in summary_rows:
    base = base_by_bucket.get(row["stress_bucket"], {})
    row["base_target_false_positive_rate"] = base.get("target_false_positive_rate")
    row["base_malformed_rate"] = base.get("malformed_rate")
    row["delta_tfpr_vs_base"] = None if row["target_false_positive_rate"] is None or base.get("target_false_positive_rate") is None else row["target_false_positive_rate"] - base["target_false_positive_rate"]
    row["delta_malformed_vs_base"] = None if row["malformed_rate"] is None or base.get("malformed_rate") is None else row["malformed_rate"] - base["malformed_rate"]
    row["stress_tfpr_budget"] = (base.get("target_false_positive_rate") or 0.0) + 0.03
    row["stress_malformed_budget"] = max(0.05, (base.get("malformed_rate") or 0.0) + 0.02)
    row["tfpr_pass"] = row["target_false_positive_rate"] is not None and row["target_false_positive_rate"] <= row["stress_tfpr_budget"]
    row["malformed_pass"] = row["malformed_rate"] is not None and row["malformed_rate"] <= row["stress_malformed_budget"]
    row["gate_activation_pass"] = row["label"] == "base" or (row["gate_activation_rate"] is not None and row["gate_activation_rate"] >= 0.95)

tradeoff_rows = []
for spec in RUN_SPECS:
    label = spec["label"]
    template = per_label_bucket[label].get("same_subject_template", {})
    generation = per_label_bucket[label].get("generation", {})
    dense = dense_by_label.get(label, {})
    malformed_pass = bool(template.get("malformed_pass")) and bool(generation.get("malformed_pass", True))
    gate_pass = bool(template.get("gate_activation_pass")) and bool(generation.get("gate_activation_pass", True))
    row = {
        "label": label,
        "family": spec["family"],
        "guidance_scale": spec["guidance_scale"],
        "rewrite_exact": dense.get("rewrite_exact"),
        "declarative_paraphrases_exact": dense.get("declarative_paraphrases_exact"),
        "selection_score": dense.get("selection_score"),
        "primary_sparse_guidance_kl": dense.get("primary_sparse_guidance_kl"),
        "gpu_minutes_per_edit": template.get("gpu_minutes_per_edit"),
        "same_subject_template_tfpr": template.get("target_false_positive_rate"),
        "same_subject_template_delta_tfpr_vs_base": template.get("delta_tfpr_vs_base"),
        "generation_tfpr": generation.get("target_false_positive_rate"),
        "generation_delta_tfpr_vs_base": generation.get("delta_tfpr_vs_base"),
        "malformed_rate": max(
            value for value in [template.get("malformed_rate"), generation.get("malformed_rate")]
            if value is not None
        ) if template or generation else None,
        "same_subject_template_tfpr_pass": template.get("tfpr_pass"),
        "generation_tfpr_pass": generation.get("tfpr_pass"),
        "malformed_pass": malformed_pass,
        "gate_activation_pass": gate_pass,
    }
    row["template_stress_pass"] = bool(row["same_subject_template_tfpr_pass"]) and malformed_pass and gate_pass
    row["strict_stress_pass"] = bool(row["same_subject_template_tfpr_pass"]) and bool(row["generation_tfpr_pass"]) and malformed_pass and gate_pass
    tradeoff_rows.append(row)

MIN_REWRITE_EXACT = 0.25
MIN_DECLARATIVE_PARAPHRASE_EXACT = 0.20
MIN_SELECTION_SCORE = 0.30

def useful_edit(row):
    try:
        rewrite = float(row.get("rewrite_exact") or 0.0)
        para = float(row.get("declarative_paraphrases_exact") or 0.0)
        score = float(row.get("selection_score") or 0.0)
    except Exception:
        return False
    return (
        rewrite >= MIN_REWRITE_EXACT
        and para >= MIN_DECLARATIVE_PARAPHRASE_EXACT
        and score >= MIN_SELECTION_SCORE
    )

selection_rows = []
for row in tradeoff_rows:
    edit_useful = useful_edit(row)
    status = "keep_dev_candidate" if row["template_stress_pass"] and edit_useful else "reject_or_diagnostic"
    selection_rows.append({
        **row,
        "min_rewrite_exact": MIN_REWRITE_EXACT,
        "min_declarative_paraphrases_exact": MIN_DECLARATIVE_PARAPHRASE_EXACT,
        "min_selection_score": MIN_SELECTION_SCORE,
        "edit_useful_pass": edit_useful,
        "decision_status": status,
    })
selection_rows.sort(key=lambda row: (row["decision_status"] != "keep_dev_candidate", -(float(row["selection_score"]) if row.get("selection_score") else 0.0)))

comparison_pairs = [
    ("myopic_score_gated_subject_gs1.5", "base"),
    ("myopic_score_gated_subject_gs1.75", "base"),
    ("mc_bridge_gated_subject_gs1.75", "base"),
    ("no_rollout_bridge_gated_subject_gs1.75", "base"),
    ("mc_bridge_gated_subject_gs1.75", "myopic_score_gated_subject_gs1.5"),
    ("mc_bridge_gated_subject_gs1.75", "myopic_score_gated_subject_gs1.75"),
    ("mc_bridge_gated_subject_gs1.75", "no_rollout_bridge_gated_subject_gs1.75"),
    ("mc_bridge_gated_subject_gs1.5", "myopic_score_gated_subject_gs1.5"),
    ("mc_bridge_gated_subject_gs1.5", "no_rollout_bridge_gated_subject_gs1.5"),
    ("no_rollout_bridge_gated_subject_gs1.5", "myopic_score_gated_subject_gs1.5"),
    ("mc_bridge_gated_subject_gs2.0", "myopic_score_gated_subject_gs2.0"),
]
metric_map = {
    "target_false_positive_rate": "target_false_positive_rate",
    "target_new_exact_rate": "exact_rate",
    "target_new_token_f1": "token_f1",
    "malformed_rate": "malformed_rate",
}
bootstrap_rows = []
for candidate, baseline in comparison_pairs:
    pair_rows = method_rows[candidate] + method_rows[baseline]
    for bucket_name in ["same_subject_template", "generation"]:
        filtered = [
            row for row in pair_rows
            if row.get("bucket") == "near_locality" and stress_bucket(row.get("prompt_id")) == bucket_name
        ]
        for metric_name, row_metric in metric_map.items():
            boot = paired_bootstrap_delta_by_case(
                filtered,
                candidate_method=candidate,
                baseline_method=baseline,
                bucket="near_locality",
                metric=row_metric,
                samples=1000,
                seed=0,
            )
            if boot:
                bootstrap_rows.append({
                    "candidate": candidate,
                    "baseline": baseline,
                    "stress_bucket": bucket_name,
                    "metric": metric_name,
                    "mean_delta_candidate_minus_baseline": boot["mean_delta"],
                    "ci_lower": boot["ci_low"],
                    "ci_upper": boot["ci_high"],
                    "num_edits": boot["num_edits"],
                })

def first_output(row):
    outputs = row.get("sample_outputs") or []
    return outputs[0] if outputs else ""

sample_methods = [
    "base",
    "prompt_memory_gated_subject_gs1.0",
    "myopic_score_gated_subject_gs1.5",
    "myopic_score_gated_subject_gs1.75",
    "mc_bridge_gated_subject_gs1.75",
    "no_rollout_bridge_gated_subject_gs1.75",
    "myopic_score_gated_subject_gs2.0",
    "mc_bridge_gated_subject_gs2.0",
]
rows_by_key = defaultdict(dict)
meta_by_edit = {}
for label in sample_methods:
    for row in method_rows[label]:
        if row.get("bucket") != "near_locality":
            continue
        key = (str(row.get("edit_id") or row.get("case_id")), str(row.get("prompt_id")))
        rows_by_key[key][label] = row

stress_records = [json.loads(line) for line in STRESS_INPUT.open("r", encoding="utf-8") if line.strip()]
for row in stress_records:
    meta_by_edit[row["id"]] = row

def flag(label, methods):
    row = methods.get(label)
    return bool(row and float(row.get("target_false_positive_rate") or 0.0) > 0.0)

sample_categories = defaultdict(list)
for key, methods in rows_by_key.items():
    myopic_leaks = flag("myopic_score_gated_subject_gs1.5", methods) or flag("myopic_score_gated_subject_gs1.75", methods)
    mc_leaks = flag("mc_bridge_gated_subject_gs1.75", methods)
    malformed = any(float(row.get("malformed_rate") or 0.0) > 0.0 for row in methods.values())
    if myopic_leaks:
        sample_categories["high_leakage_myopic"].append((key, methods))
    if mc_leaks:
        sample_categories["high_leakage_mc_bridge"].append((key, methods))
    if myopic_leaks and not mc_leaks:
        sample_categories["myopic_leaks_mc_safe"].append((key, methods))
    if mc_leaks and not myopic_leaks:
        sample_categories["mc_leaks_myopic_safe"].append((key, methods))
    if not any(flag(label, methods) for label in sample_methods):
        sample_categories["all_methods_safe"].append((key, methods))
    if malformed:
        sample_categories["malformed_cases"].append((key, methods))

sample_rows = []
for category, items in sample_categories.items():
    for (edit_id, prompt_id), methods in items[:20]:
        meta = meta_by_edit.get(edit_id, {})
        row = {
            "sample_category": category,
            "edit_id": edit_id,
            "subject": meta.get("subject"),
            "relation_id": meta.get("relation_id"),
            "target_new": meta.get("target"),
            "target_true": meta.get("old_target"),
            "stress_bucket": stress_bucket(prompt_id),
            "stress_prompt": next(iter(methods.values())).get("prompt") if methods else None,
            "target_new_match_flags": json.dumps({label: flag(label, methods) for label in sample_methods}, ensure_ascii=False),
        }
        for label in sample_methods:
            row[f"{label}_output"] = first_output(methods.get(label, {}))
        sample_rows.append(row)

write_csv(OUT_REPORT / "same_subject_stress_midkl_summary.csv", summary_rows)
write_csv(OUT_REPORT / "stress_vs_edit_tradeoff.csv", tradeoff_rows)
write_csv(OUT_REPORT / "same_subject_stress_midkl_paired_bootstrap.csv", bootstrap_rows)
write_csv(OUT_REPORT / "same_subject_stress_midkl_output_samples.csv", sample_rows)
write_csv(OUT_REPORT / "method_selection_after_stress.csv", selection_rows)

plot_rows = [row for row in tradeoff_rows if row.get("rewrite_exact") not in {None, ""} and row.get("same_subject_template_tfpr") is not None]
colors = {
    "prompt_memory": "#1f77b4",
    "myopic_score": "#ff7f0e",
    "no_rollout_bridge": "#2ca02c",
    "mc_bridge": "#d62728",
    "base": "#7f7f7f",
}
fig, ax = plt.subplots(figsize=(8, 5))
for row in plot_rows:
    ax.scatter(
        float(row["same_subject_template_tfpr"]),
        float(row["selection_score"]),
        color=colors.get(row["family"], "#333333"),
        label=row["family"],
    )
    ax.annotate(str(row["guidance_scale"]), (float(row["same_subject_template_tfpr"]), float(row["selection_score"])), fontsize=8)
template_budget = per_label_bucket["base"]["same_subject_template"]["target_false_positive_rate"] + 0.03
ax.axvline(template_budget, color="#555555", linestyle="--", linewidth=1)
handles, labels = ax.get_legend_handles_labels()
dedup = dict(zip(labels, handles))
ax.legend(dedup.values(), dedup.keys())
ax.set_xlabel("Same-subject template TFPR")
ax.set_ylabel("Selection score")
ax.set_title("Same-Subject Stress vs Edit Score")
fig.tight_layout()
fig.savefig(OUT_REPORT / "stress_tradeoff_plot.png", dpi=200)
plt.close(fig)

stress_counts = json.load(STRESS_SUMMARY.open())["stress_counts"]
candidate_labels = [
    "prompt_memory_gated_subject_gs1.0",
    "myopic_score_gated_subject_gs1.5",
    "myopic_score_gated_subject_gs1.75",
    "myopic_score_gated_subject_gs2.0",
    "mc_bridge_gated_subject_gs1.5",
    "mc_bridge_gated_subject_gs1.75",
    "mc_bridge_gated_subject_gs2.0",
    "no_rollout_bridge_gated_subject_gs1.5",
    "no_rollout_bridge_gated_subject_gs1.75",
    "no_rollout_bridge_gated_subject_gs2.0",
]
report = {
    "protocol_version": "counterfact_direction1_v1",
    "split_role": "dev_tune_200",
    "stress_name": "same_subject_locality_midkl",
    "gate_mode": "subject",
    "analysis_500_used": False,
    "final_test_used": False,
    "stress_input": "runs/counterfact_direction1_v1/same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl",
    "num_edits": 200,
    "stress_counts": {
        "same_subject_template": int(stress_counts.get("same_subject_template", 0)),
        "generation": int(stress_counts.get("generation", 0)),
    },
    "primary_stress_bucket": "same_subject_template",
    "secondary_stress_bucket": "generation",
    "stress_budget_rule": "stress_tfpr_budget = base_stress_tfpr + 0.03; stress_malformed_budget = max(0.05, base_stress_malformed + 0.02)",
    "candidate_labels": candidate_labels,
    "decision_rule": "A method may remain a dev candidate only if template_stress_pass is true and edit performance remains useful.",
    "edit_useful_thresholds": {
        "min_rewrite_exact": MIN_REWRITE_EXACT,
        "min_declarative_paraphrases_exact": MIN_DECLARATIVE_PARAPHRASE_EXACT,
        "min_selection_score": MIN_SELECTION_SCORE,
    },
    "artifacts": {
        "summary": str(OUT_REPORT / "same_subject_stress_midkl_summary.csv"),
        "tradeoff": str(OUT_REPORT / "stress_vs_edit_tradeoff.csv"),
        "paired_bootstrap": str(OUT_REPORT / "same_subject_stress_midkl_paired_bootstrap.csv"),
        "output_samples": str(OUT_REPORT / "same_subject_stress_midkl_output_samples.csv"),
        "method_selection": str(OUT_REPORT / "method_selection_after_stress.csv"),
        "plot": str(OUT_REPORT / "stress_tradeoff_plot.png"),
    },
}
with (OUT_REPORT / "report_summary.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Wrote:", OUT_REPORT)
print("Method selection after stress:")
for row in selection_rows:
    print(row)

## Expected Outcome

Inspect first:

```text
same_subject_stress_midkl_summary.csv
stress_vs_edit_tradeoff.csv
same_subject_stress_midkl_paired_bootstrap.csv
method_selection_after_stress.csv
stress_tradeoff_plot.png
```

Decision cases:

```text
Case A: a mid-KL method passes template stress and remains useful.
Case B: myopic passes and bridge fails.
Case C: MC bridge passes and myopic fails.
Case D: all subject-gated mid-KL methods fail, so the subject gate should be rejected.
```

Do not move to `analysis_500` after this notebook. If a mid-KL candidate survives, the next step is still an edit-intent or hybrid gate check before locking a dev method.

The decision file now requires both `template_stress_pass` and `edit_useful_pass` before a method can remain a dev candidate.
